In [29]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

import pickle

from sklearn.metrics import root_mean_squared_error

In [2]:
def read_parquet(file_link):
    df = pd.read_parquet(file_link)

    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[df.duration >= 1]
    df = df[df.duration <= 60]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [3]:
df_train = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_eval = read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [19]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_eval['PU_DO'] = df_eval['PULocationID'] + '_' + df_eval['DOLocationID']

In [21]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

eval_dicts = df_eval[categorical + numerical].to_dict(orient='records')
X_eval = dv.transform(eval_dicts)

In [22]:
import numpy as np

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_eval.indices = X_eval.indices.astype(np.int32)
X_eval.indptr = X_eval.indptr.astype(np.int32)

In [23]:
target = 'duration'
y_train = df_train[target].values
y_eval = df_eval[target].values

In [24]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

7.758715206476337

In [28]:
lr = Lasso(alpha=0.001)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

9.23343622815571

In [30]:
lr = Ridge()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)

root_mean_squared_error(y_eval, y_pred)

7.70373514032001